In [ ]:
import os
import sys
import random
import numpy as np
import cv2
import torch
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

In [ ]:
def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

In [ ]:
@dataclass
class CFG:
    batch_size : int = 32
    seed: int = 42
    early_stopping_steps :int = 9
    number_of_channels : int = 3
    steps_until_plot: int = 2
    criterion : nn.Module = nn.CrossEntropyLoss()
    epochs : int = 80
    device : str = 'mps' if torch.backends.mps.is_available() else 'cpu'
    img_size : int = 64
    base_path : str = 'land_patches'
    number_of_pictures_example: int = 5
    train_path : str = os.path.join(base_path, 'train')
    test_path : str = os.path.join(base_path, 'test')
    validation_path : str = os.path.join(base_path, 'val')
    num_classes = len(os.listdir(train_path))
    image_extension : str = '.jpg'
    mapping_between_folders_and_classes = {
        '0': 'AnnualCrop',
        '1': 'Forest',
        '2': 'HerbaceousVegetation',
        '3': 'Highway',
        '4': 'Industrial',
        '5': 'Pasture',
        '6': 'PermanentCrop',
        '7': 'Residential',
        '8': 'River',
        '9': 'SeaLake'
    }
    num_classes : int = len(mapping_between_folders_and_classes)
    imagenet_mean  = [0.485, 0.456, 0.406]
    imagenet_std  = [0.229, 0.224, 0.225]

In [ ]:
def see_class_distribution_for_test_train_val(path):
  class_distribution = {}

  for class_folder in sorted(os.listdir(path)):
    class_path = os.path.join(path, class_folder)
    if os.path.isdir(class_path):
      num_images = len([f for f in os.listdir(class_path) if f.endswith(CFG.image_extension)])
      class_distribution[class_folder] = num_images

  return class_distribution

In [ ]:
def plot_class_distribution_by_class(class_distribution, filename=None):
  plt.figure(figsize=(10, 5))
  classes = list(class_distribution.keys())
  counts = [class_distribution[c] for c in classes]
  plt.bar(classes, counts, color='steelblue', edgecolor='black')
  plt.xlabel('Clasa', fontsize=12)
  plt.ylabel('Număr de imagini', fontsize=12)
  plt.title('Distribuția Claselor în Setul de Antrenare', fontsize=14, fontweight='bold')
  plt.grid(axis='y', alpha=0.3)
  plt.tight_layout()
  if filename:
      plt.savefig(filename)
      plt.close()
  else:
      plt.show()

In [ ]:
train_distribution = see_class_distribution_for_test_train_val(path=CFG.train_path)
test_distribution = see_class_distribution_for_test_train_val(path=CFG.test_path)
validation_distribution = see_class_distribution_for_test_train_val(path=CFG.validation_path)

In [ ]:
plot_class_distribution_by_class(train_distribution, filename='land_patches_train_dist.png')
plot_class_distribution_by_class(test_distribution, filename='land_patches_test_dist.png')
plot_class_distribution_by_class(validation_distribution, filename='land_patches_val_dist.png')

In [ ]:
def is_class_balanced(class_distribution):
  is_balanced = len(set(class_distribution.values())) == 1
  print(f"Dataset-ul este {'Echilibrat' if is_balanced else 'Dezechilibrat'}")

In [ ]:
is_class_balanced(train_distribution)
is_class_balanced(test_distribution)
is_class_balanced(validation_distribution)

In [ ]:
def get_parameters_for_type_of_operation(type = 'picture_stats'):
    if type == 'picture_stats':
        return [],{}
    elif type == 'correlation_purpose':
        return [],[]
    elif type == 'plot_images':
        return [],[]
    elif type == 'make_csv_files':
        return [],[]

    return None

In [ ]:
def get_random_images_to_plot(all_images):
    random.seed(CFG.seed)
    return random.sample(all_images, min(CFG.number_of_pictures_example, len(all_images)))

In [ ]:
def set_path(train=True,validation=None):
    if validation:
      return CFG.validation_path if train else CFG.test_path
    return CFG.train_path if train else CFG.test_path

In [ ]:
def get_path(class_idx, train=True, validation=None):
    base_path = set_path(train, validation)
    all_items = sorted(os.listdir(base_path))
    valid_folders = [
        d for d in all_items
        if os.path.isdir(os.path.join(base_path, d)) and not d.startswith('.')
    ]

    if 0 <= class_idx < len(valid_folders):
        return os.path.join(base_path, valid_folders[class_idx])
    return None

In [ ]:
def get_all_images(class_idx, train=True, validation=None):
    class_path = get_path(class_idx, train, validation)
    if class_path is None:
        return []
    return [f for f in os.listdir(class_path) if f.endswith(CFG.image_extension)]

In [ ]:
def read_images(image_name, class_idx, train=True, validation=None):
    img_path = os.path.join(get_path(class_idx, train, validation), image_name)
    return img_path, cv2.imread(img_path)

In [ ]:
def prepare_plot():
    fig, axes = plt.subplots(CFG.num_classes, CFG.number_of_pictures_example, figsize=(16, 20))
    fig.suptitle('Exemple din Fiecare Clasă - Variabilitate Intra-Clasă', fontsize=16, fontweight='bold')
    return fig, axes

In [ ]:
def plot_examples_from_dataset(image, axes, class_idx, img_idx):
    ax = axes[class_idx, img_idx]
    ax.imshow(image)
    ax.axis('off')

    if img_idx == 0:
        ax.set_title(f'Clasa {class_idx}', fontsize=12, fontweight='bold', loc='left')

In [ ]:
def process_all_images(number_of_classes=CFG.num_classes, operation_type='picture_stats', train=True, validation=None):
    container1, container2 = get_parameters_for_type_of_operation(operation_type)

    if operation_type == 'plot_images':
        _, axes = prepare_plot()

    for class_idx in range(number_of_classes):
        all_images = get_all_images(class_idx, train=train, validation=validation)

        if operation_type == 'plot_images':
            all_images = get_random_images_to_plot(all_images)

        class_pixels = []
        for img_name in all_images:
            img_path, image = read_images(img_name, class_idx, train=train, validation=validation)

            if image is None:
                continue

            if operation_type == 'picture_stats':
                container1.append(image.shape[:2])
                class_pixels.append(np.array(image))

            elif operation_type == 'correlation_purpose':
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                vector = image_rgb.flatten()
                container1.append(vector)
                container2.append(class_idx)

            elif operation_type == 'plot_images':
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                plot_examples_from_dataset(image_rgb, axes, class_idx, img_idx=all_images.index(img_name))

            elif operation_type == 'make_csv_files':
                container1.append({'ID': img_path, 'label': class_idx})

        if operation_type == 'picture_stats' and class_pixels:
            class_pixels_np = np.array(class_pixels)
            container2[class_idx] = {
                'mean': np.mean(class_pixels_np),
                'std': np.std(class_pixels_np),
                'min': np.min(class_pixels_np),
                'max': np.max(class_pixels_np)
            }

    if operation_type == 'correlation_purpose':
        return np.array(container1), np.array(container2)

    if operation_type == 'plot_images':
        plt.tight_layout()
        plt.show()

    return container1, container2

In [ ]:
image_sizes, pixel_stats_per_class = process_all_images()
print(f"\n{'Dimensiuni imagini:':<25} {Counter(image_sizes).most_common(1)[0][0]}")

print("-" * 60)
print("Statistici pixeli per clasă (Media ± Deviație Standard):")
print("-" * 60)

for class_idx in range(1, CFG.num_classes):
    stats = pixel_stats_per_class[class_idx-1]
    print(f"Clasa {class_idx:2d}: {stats['mean']:6.2f} ± {stats['std']:6.2f}  (min: {stats['min']:3.0f}, max: {stats['max']:3.0f})")

print("=" * 60)

In [ ]:
def plot_rgb_distribution_per_class():
    _, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for class_idx in range(1, CFG.num_classes + 1):
        all_images = get_all_images(class_idx, train=True)
        sample_images = get_random_images_to_plot(all_images) if len(all_images) > 50 else all_images
        
        r_vals, g_vals, b_vals = [], [], []
        
        for img_name in sample_images:
            _, image = read_images(img_name, class_idx, train=True)
            if image is not None:
                img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                r_vals.extend(img_rgb[:,:,0].flatten())
                g_vals.extend(img_rgb[:,:,1].flatten())
                b_vals.extend(img_rgb[:,:,2].flatten())
        
        ax = axes[class_idx - 1]
        sns.kdeplot(r_vals, color='red', ax=ax, fill=True, alpha=0.1, label='R')
        sns.kdeplot(g_vals, color='green', ax=ax, fill=True, alpha=0.1, label='G')
        sns.kdeplot(b_vals, color='blue', ax=ax, fill=True, alpha=0.1, label='B')
        
        ax.set_title(f'Class {class_idx} Color Distribution')
        if class_idx == 1: ax.legend()

    plt.tight_layout()
    plt.show()

plot_rgb_distribution_per_class()

In [ ]:
def plot_average_images():
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for class_idx in range(1, CFG.num_classes + 1):
        all_images = get_all_images(class_idx, train=True)
        sum_image = None
        count = 0
        
        for img_name in all_images:
            img_path, image = read_images(img_name, class_idx, train=True)
            if image is not None:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                image = cv2.resize(image, (CFG.img_size, CFG.img_size))
                
                if sum_image is None:
                    sum_image = np.float32(image)
                else:
                    sum_image += image
                count += 1
        
        if count > 0:
            avg_image = sum_image / count
            avg_image = np.array(np.round(avg_image), dtype=np.uint8)
            
            axes[class_idx-1].imshow(avg_image)
            axes[class_idx-1].set_title(f"Avg: Class {class_idx}")
            axes[class_idx-1].axis('off')
            
    plt.suptitle("Average Image per Class (Canonical Representation)", fontsize=16)
    plt.tight_layout()
    plt.show()

plot_average_images()

In [ ]:
X, y = process_all_images(operation_type = 'correlation_purpose')
corr_matrix = np.corrcoef(X)

intra_correlations = []
inter_correlations = []
num_samples = len(y)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

class_mask = (y[:, None] == y[None, :]) & mask

intra_correlations = corr_matrix[class_mask]
inter_correlations = corr_matrix[mask & ~class_mask]

avg_intra = np.mean(intra_correlations)
avg_inter = np.mean(inter_correlations)

print(f"Average Intra-class Correlation (Consistency): {avg_intra:.4f}")
print(f"Average Inter-class Correlation (Confusion):   {avg_inter:.4f}")
print(f"Separability Score (Intra - Inter):            {avg_intra - avg_inter:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.kdeplot(intra_correlations, label='Intra-class', fill=True, color='green')
sns.kdeplot(inter_correlations, label='Inter-class', fill=True, color='red')
plt.xlabel('Correlation Coefficient')
plt.legend()

In [ ]:
_, _ = process_all_images(operation_type = 'plot_images')

In [ ]:
def make_train_and_test_csvs(train=True, validation=None):
    data, _ = process_all_images(operation_type='make_csv_files', train=train, validation=validation)
    return pd.DataFrame(data)

In [ ]:
def locate_csv_to_folders(train=True, validation=None):
    df = make_train_and_test_csvs(train=train, validation=validation)
    if validation:
      split_name = "validation"
    else:
      split_name = "train" if train else "test"
    csv_path = os.path.join(set_path(train, validation), f"{split_name}_data.csv")
    df.to_csv(csv_path, index=False)

In [ ]:
locate_csv_to_folders(train=True)
locate_csv_to_folders(train=False)
locate_csv_to_folders(train=True,validation=True)

In [ ]:
train_set = pd.read_csv(f'{CFG.train_path}/train_data.csv')
val_set = pd.read_csv(f'{CFG.validation_path}/validation_data.csv')
test_set = pd.read_csv(f'{CFG.test_path}/test_data.csv')

In [ ]:
class EarlyStopping:
    def __init__(self, patience=CFG.early_stopping_steps, verbose=False, path='checkpoint.pt', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False
        self.path = path
        self.trace_func = trace_func

    def __call__(self, current_accuracy, model):
        if current_accuracy <= self.best_score + 1e-6:
            self.counter += 1
            if self.verbose:
                self.trace_func(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            if self.verbose:
                self.trace_func(
                    f"Validation accuracy increased ({self.best_score:.6f} → {current_accuracy:.6f}). Saving model..."
                )
            self.save_checkpoint(model)
            self.best_score = current_accuracy
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)


In [ ]:
class ImprovedMLP(nn.Module):
    def __init__(self, input_size = CFG.number_of_channels * CFG.img_size * CFG.img_size,
                 hidden_sizes=[512, 256, 128], num_classes=CFG.num_classes, dropout_rate=0.2):
        super(ImprovedMLP, self).__init__()

        layers = []
        in_features = input_size

        for i, hidden in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_features, hidden))
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            dropout = dropout_rate if i < len(hidden_sizes) - 1 else dropout_rate * 0.5
            layers.append(nn.Dropout(dropout))
            in_features = hidden

        layers.append(nn.Linear(in_features, num_classes))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.4),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [ ]:
class MyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

        self.resize = transforms.Resize((CFG.img_size, CFG.img_size))
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=CFG.imagenet_mean, std=CFG.imagenet_std)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['ID']
        label = self.df.iloc[idx]['label']
        image = Image.open(f'{img_name}').convert('RGB')

        if self.transform:
            image = self.resize(image)
            image = self.transform(image)
            image = self.normalize(image)
        else:
            image = self.resize(image)
            image = self.normalize(image)

        return image, label


train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.RandomVerticalFlip(p=0.4),
])

train_dataset = MyDataset(train_set, transform=train_transform)
train_dataset_without_augmentations = MyDataset(train_set,transform=None)
validation_dataset = MyDataset(val_set)
test_dataset = MyDataset(test_set)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True)
train_loader_without_augmentations = DataLoader(train_dataset_without_augmentations,batch_size=CFG.batch_size,shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False)

In [ ]:
def get_number_of_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def top_k_accuracy(k, target, output, device):
    batch_size = target.size(0)
    
    _, pred = output.topk(k, 1, True, True)
    
    pred = pred.t()
    
    correct = pred.eq(target.to(device).view(1, -1).expand_as(pred))
    
    correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
    correct_k = correct_k.mul_(100.0 / batch_size)
    
    return correct_k.item()

In [ ]:
def plot_loss_curves(train_losses, val_losses, filename=None):
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=5)
    plt.plot(epochs, val_losses, 'r-o', label='Validation Loss', linewidth=2, markersize=5)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.title('Curbe de Loss pentru Antrenare și Validare', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if filename:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()

In [ ]:
def plot_accuracy_curves(train_top1, val_top1, filename=None):
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_top1) + 1)
    
    plt.plot(epochs, train_top1, 'b-o', label='Training Top-1 Accuracy', linewidth=2, markersize=5)
    plt.plot(epochs, val_top1, 'r-o', label='Validation Top-1 Accuracy', linewidth=2, markersize=5)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Acuratețe (%)', fontsize=14)
    plt.title('Curbe de Acuratețe pentru Antrenare și Validare', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if filename:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, filename=None):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=range(1, CFG.num_classes + 1),
                yticklabels=range(1, CFG.num_classes + 1),
                cbar_kws={'label': 'Număr de predicții'})
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.title('Matricea de Confuzie', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if filename:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()
    
    return cm

In [ ]:
def validate(model, criterion, val_loader, device, epoch):
    model.eval()
    running_loss = 0.0
    running_top1_acc = 0.0

    bar = tqdm(enumerate(val_loader), total=len(val_loader), colour='green', file=sys.stdout)

    with torch.no_grad():
        for i, (images, labels) in bar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels.long())

            top1_acc = top_k_accuracy(1, labels, outputs, device)

            running_loss += loss.item() * images.size(0)
            running_top1_acc += top1_acc * images.size(0)

            avg_loss = running_loss / ((i + 1) * images.size(0))
            avg_top1 = running_top1_acc / ((i + 1) * images.size(0))

            bar.set_postfix({
                'epoch': epoch,
                'val_loss': f'{avg_loss:.4f}',
                'top1_accuracy': f'{avg_top1:.2f}%'
            })

    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_top1 = running_top1_acc / len(val_loader.dataset)

    return epoch_loss, epoch_top1

In [ ]:
def train_single_model(class_name, config, train_loader, val_loader, device, run_prefix, model_idx, num_epochs=CFG.epochs):
    model = class_name().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], 
                           weight_decay=config['weight_decay'], betas=(0.9, 0.999))
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = CFG.criterion
    
    checkpoint_path = f"{run_prefix}_model_{model_idx}_best.pth"
    early_stopping = EarlyStopping(patience=CFG.early_stopping_steps, verbose=True, path=checkpoint_path)
    
    train_losses, train_top1 = [], []
    val_losses, val_top1 = [], []
    
    best_val_acc = 0.0
    best_epoch = 0
    
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        running_top1_acc = 0.0
        
        bar = tqdm(enumerate(train_loader), total=len(train_loader), colour='cyan', file=sys.stdout)

        for i, (images, labels) in bar:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()
            
            top1_acc = top_k_accuracy(1, labels, outputs, device)
            
            running_loss += loss.item() * images.size(0)
            running_top1_acc += top1_acc * images.size(0)

            avg_loss = running_loss / ((i + 1) * images.size(0))
            avg_top1 = running_top1_acc / ((i + 1) * images.size(0))

            bar.set_postfix({
            'epoch': epoch,
            'train_loss': f'{avg_loss:.4f}',
            'top1_accuracy': f'{avg_top1:.2f}%'
            })
            
        scheduler.step()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_top1 = running_top1_acc / len(train_loader.dataset)

        print(f"Train - Loss: {epoch_loss:.4f}, Top-1: {epoch_top1:.2f}%")

        train_losses.append(epoch_loss)
        train_top1.append(epoch_top1)
        
        val_loss, val_acc = validate(model, criterion, val_loader, device, epoch)

        print(f"Val - Loss: {val_loss:.4f}, Val - Top-1: {val_acc:.2f}%")

        val_losses.append(val_loss)
        val_top1.append(val_acc)
        
        early_stopping(val_acc, model)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
        
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break

        if epoch % CFG.steps_until_plot == 0:
            plot_loss_curves(train_losses, val_losses)
            plot_accuracy_curves(train_top1, val_top1)
            pass

    if best_epoch > 0:
        final_train_losses = train_losses[:best_epoch]
        final_val_losses = val_losses[:best_epoch]
        final_train_top1 = train_top1[:best_epoch]
        final_val_top1 = val_top1[:best_epoch]
        
        loss_filename = f"{run_prefix}_model_{model_idx}_loss_curve.png"
        acc_filename = f"{run_prefix}_model_{model_idx}_acc_curve.png"
        
        plot_loss_curves(final_train_losses, final_val_losses, filename=loss_filename)
        plot_accuracy_curves(final_train_top1, final_val_top1, filename=acc_filename)
        print(f"Saved best plots up to epoch {best_epoch} to {loss_filename} and {acc_filename}")

    return model, best_val_acc

In [ ]:
def predict_soft_vote_ensemble(models, loader, device):
    for m in models:
        m.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            avg_probs = torch.zeros(images.size(0), CFG.num_classes).to(device)

            for model in models:
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                avg_probs += probs

            avg_probs /= len(models)

            _, predicted = torch.max(avg_probs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
def predict_hard_voting_ensemble(models, loader, device):
    for m in models:
        m.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            model_predictions  = []

            for model in models:
                outputs = model(images)
                _, predicted = outputs.max(1)
                model_predictions.append(predicted.cpu().numpy())

            model_predictions = np.array(model_predictions)

            ensemble_pred = []
            for i in range(model_predictions.shape[1]):
                votes = model_predictions[:, i]
                majority = Counter(votes).most_common(1)[0][0]
                ensemble_pred.append(majority)

            all_preds.extend(ensemble_pred)
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
def apply_tta_augmentations(img_tensor):
    augmentations = []
    
    augmentations.append(img_tensor)

    augmentations.append(torch.flip(img_tensor, dims=[2]))

    augmentations.append(torch.flip(img_tensor, dims=[1]))

    augmentations.append(torch.rot90(img_tensor, 1, [1, 2]))
    
    return augmentations

In [ ]:
def predict_ensemble_with_tta(models, loader, device, num_tta=2):
    for m in models:
        m.eval()
        
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="TTA Ensemble Prediction", colour='magenta'):
            images, labels = images.to(device), labels.to(device)
            batch_size = images.size(0)
            
            final_probs = torch.zeros(batch_size, CFG.num_classes).to(device)
            
            for img_idx in range(batch_size):
                img = images[img_idx]
                
                
                augmented_images = apply_tta_augmentations(img)
                augmented_images = augmented_images[:num_tta]
                
                aug_batch = torch.stack(augmented_images)
                
                tta_probs = torch.zeros(CFG.num_classes).to(device)
                
                for model in models:
                    outputs = model(aug_batch)
                    probs = F.softmax(outputs, dim=1)
                    avg_probs = probs.mean(dim=0)
                    tta_probs += avg_probs
                
                tta_probs /= len(models)
                final_probs[img_idx] = tta_probs
            
            _, predicted = torch.max(final_probs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return np.array(all_preds), np.array(all_labels)


In [ ]:
def calculate_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred, average='weighted') * 100

In [ ]:
def run_models_ensemble_training(model_class, train_loader, validation_loader,ensemble_configs, run_prefix):
    trained_models = []
    model_accuracies = []

    for i, conf in enumerate(ensemble_configs):
        print(f"\n{'='*60}")
        print(f"Training Model {i+1}/{len(ensemble_configs)}")
        print(f"Config: lr={conf['lr']}, weight_decay={conf['weight_decay']}")
        print(f"{'='*60}\n")
        
        model, best_acc = train_single_model(model_class, conf, train_loader, validation_loader, CFG.device, run_prefix, i+1)
        trained_models.append(model)
        model_accuracies.append(best_acc)
        
        print(f"\nModel {i+1} Best Validation Accuracy: {best_acc:.2f}%")

    print(f"\n{'='*60}")
    print("Ensemble Training Complete!")
    print(f"{'='*60}")
    print("\nIndividual Model Accuracies:")
    for i, acc in enumerate(model_accuracies):
        print(f"  Model {i+1}: {acc:.2f}%")
    print(f"\nAverage Accuracy: {np.mean(model_accuracies):.2f}%")
    
    return trained_models, model_accuracies

In [ ]:
ensemble_configs = [
    {'lr': 3e-3, 'weight_decay': 1e-4},
    {'lr': 3e-3, 'weight_decay': 2e-4},
    {'lr': 3e-3, 'weight_decay': 3e-4},
    {'lr': 3e-3, 'weight_decay': 1.5e-4},
    {'lr': 3e-3, 'weight_decay': 0.9e-4}
]

In [ ]:
trained_models_for_mlp_without_augmentations, accuracies_for_mlp_without_augmentations = run_models_ensemble_training(
    ImprovedMLP, train_loader_without_augmentations, validation_loader, ensemble_configs, "LandPatches_MLP_NoAug"
)
trained_models_for_cnn_without_augmentations, accuracies_for_cnn_without_augmentations = run_models_ensemble_training(
    CNN, train_loader_without_augmentations, validation_loader, ensemble_configs, "LandPatches_CNN_NoAug"
)

trained_models_for_mlp_with_augmentations, accuracies_for_mlp_with_augmentations = run_models_ensemble_training(
    ImprovedMLP, train_loader, validation_loader, ensemble_configs, "LandPatches_MLP_Aug"
)
trained_models_for_cnn_with_augmentations, accuracies_for_cnn_with_augmentations = run_models_ensemble_training(
    CNN, train_loader, validation_loader, ensemble_configs, "LandPatches_CNN_Aug"
)

In [ ]:
def load_models_from_checkpoints(model_class, checkpoint_pattern, num_models=5):
    loaded_models = []
    
    for i in range(1, num_models + 1):
        checkpoint_path = checkpoint_pattern.format(i)
        
        if not os.path.exists(checkpoint_path):
            print(f"Warning: Checkpoint {checkpoint_path} not found, skipping...")
            continue
            
        model = model_class().to(CFG.device)
        model.load_state_dict(torch.load(checkpoint_path, map_location=CFG.device))
        model.eval()
        
        loaded_models.append(model)
        print(f"Loaded model from {checkpoint_path}")
    
    return loaded_models

trained_models_for_mlp_with_augmentations = load_models_from_checkpoints(
    ImprovedMLP, 
    "LandPatches_MLP_Aug_model_{}_best.pth",
    num_models=5
)

trained_models_for_cnn_with_augmentations = load_models_from_checkpoints(
    CNN,
    "LandPatches_CNN_Aug_model_{}_best.pth", 
    num_models=5
)

In [ ]:
def compute_validation_accuracies(models, val_loader, device):
    accuracies = []
    
    for i, model in enumerate(models):
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Evaluating model {i+1}"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        acc = 100 * correct / total
        accuracies.append(acc)
        print(f"Model {i+1} Validation Accuracy: {acc:.2f}%")
    
    return accuracies

accuracies_for_mlp_with_augmentations = compute_validation_accuracies(
    trained_models_for_mlp_with_augmentations, 
    validation_loader, 
    CFG.device
)

accuracies_for_cnn_with_augmentations = compute_validation_accuracies(
    trained_models_for_cnn_with_augmentations,
    validation_loader,
    CFG.device
)

In [ ]:
def add_to_summary(model_type, augmentation, accuracies):
    summary_data = []
    for i, acc in enumerate(accuracies):
        summary_data.append({
            'Model Type': model_type,
            'Augmentation': augmentation,
            'Model Index': i + 1,
            'Accuracy': acc
        })

    return summary_data

all_summary_data = []
all_summary_data.extend(add_to_summary('MLP', 'No', accuracies_for_mlp_without_augmentations))
all_summary_data.extend(add_to_summary('CNN', 'No', accuracies_for_cnn_without_augmentations))
all_summary_data.extend(add_to_summary('MLP', 'Yes', accuracies_for_mlp_with_augmentations))
all_summary_data.extend(add_to_summary('CNN', 'Yes', accuracies_for_cnn_with_augmentations))

summary_df = pd.DataFrame(all_summary_data)
print("\nSummary of Model Accuracies:")
print(summary_df)

summary_df.to_csv('land_patches_model_accuracies_summary.csv', index=False)
print("Summary table saved to 'land_patches_model_accuracies_summary.csv'")

In [ ]:
def get_top_k_models_to_test(trained_models, model_accuracies, k=3):
    combined = list(zip(trained_models, model_accuracies))
    sorted_models = sorted(combined, key=lambda x: x[1], reverse=True)
    top_k = sorted_models[:k]
    top_k_models = [m for m, _ in top_k]
    top_k_accuracies = [acc for _, acc in top_k]
    return top_k_models, top_k_accuracies

In [ ]:
def predict_on_ensemble_with_TTA_for_k_models(top_k_models, test_loader, device, run_prefix=None, k=None):
    print("\nEvaluating Ensemble with TTA...")
    results = []
    y_pred_ens_tta = None
    y_true_ens_tta = None
    
    for i in range(1,5):
        print(f" - Using {i} TTA augmentations...")
        y_pred_ens_tta, y_true_ens_tta = predict_ensemble_with_tta(top_k_models, test_loader, device, num_tta=i)
        
        ens_tta_acc_i = accuracy_score(y_true_ens_tta, y_pred_ens_tta) * 100
        ens_tta_f1_i = f1_score(y_true_ens_tta, y_pred_ens_tta, average='weighted') * 100
    
        print(f"   Ensemble with {i} TTA Accuracy: {ens_tta_acc_i:.2f}%")
        print(f"   Ensemble with {i} TTA F1 Score: {ens_tta_f1_i:.2f}%")

        results.append({
            'Method': f'SoftVote_TTA_{i}',
            'Accuracy': ens_tta_acc_i,
            'F1_Score': ens_tta_f1_i
        })

        if run_prefix and k is not None:
             cm_filename = f"{run_prefix}_Top{k}_Ensemble_TTA_{i}_CM.png"
             plot_ensemble_confusion_matrix(y_true_ens_tta, y_pred_ens_tta, filename=cm_filename)
             print(f"   Saved confusion matrix to {cm_filename}")
    
    return y_pred_ens_tta, y_true_ens_tta, results

In [ ]:
def predict_on_ensemble_hard_voting_for_k_models(top_k_models, test_loader, device):
    print("\nEvaluating Ensemble with Hard Voting...")
    y_pred_ens_hard, y_true_ens_hard = predict_hard_voting_ensemble(top_k_models, test_loader, device)
    
    ens_hard_acc = accuracy_score(y_true_ens_hard, y_pred_ens_hard) * 100
    ens_hard_f1 = f1_score(y_true_ens_hard, y_pred_ens_hard, average='weighted') * 100

    print(f"   Ensemble Hard Voting Accuracy: {ens_hard_acc:.2f}%")
    print(f"   Ensemble Hard Voting F1 Score: {ens_hard_f1:.2f}%")
    
    result = {
        'Method': 'HardVoting',
        'Accuracy': ens_hard_acc,
        'F1_Score': ens_hard_f1
    }
    
    return y_pred_ens_hard, y_true_ens_hard, result

In [ ]:
def plot_ensemble_confusion_matrix(y_true, y_pred, filename=None):
    cm = plot_confusion_matrix(y_true, y_pred, filename=filename)
    return cm

In [ ]:
def print_ensemble_results(trained_models, model_accuracies, test_loader, device, type='MLP', aug=True):
    aug_str = "Aug" if aug else "NoAug"
    run_prefix = f"{type}_{aug_str}"
    
    all_k_results = []
    
    for k in range(len(ensemble_configs)):
        top_k = k + 1
        top_k_models, _ = get_top_k_models_to_test(trained_models, model_accuracies, top_k)
        print(f"\n{'='*60}")
        print(f"Evaluating Top {top_k} {type} Models Ensemble ({aug_str})")
        print(f"{'='*60}")
        
        _, _, tta_results = predict_on_ensemble_with_TTA_for_k_models(top_k_models, test_loader, device, run_prefix=run_prefix, k=top_k)
        
        for res in tta_results:
            res.update({
                'Model Type': type,
                'Augmentation': 'Yes' if aug else 'No',
                'Ensemble Size': top_k
            })
            all_k_results.append(res)
        
        y_pred_ens_hard, y_true_ens_hard, hard_res = predict_on_ensemble_hard_voting_for_k_models(top_k_models, test_loader, device)
        
        hard_res.update({
                'Model Type': type,
                'Augmentation': 'Yes' if aug else 'No',
                'Ensemble Size': top_k
        })
        all_k_results.append(hard_res)
        
        hard_cm_filename = f"{run_prefix}_Top{top_k}_Ensemble_HardVoting_CM.png"
        print(f"\n{'='*60}")
        print(f"Confusion Matrix for Top {top_k} {type} Models Ensemble with Hard Voting")
        print(f"{'='*60}")
        plot_ensemble_confusion_matrix(y_true_ens_hard, y_pred_ens_hard, filename=hard_cm_filename)
        print(f"Saved Hard Voting CM to {hard_cm_filename}")
        print("\n")
        
    return all_k_results

In [ ]:
final_results = []

final_results.extend(print_ensemble_results(trained_models_for_mlp_with_augmentations, accuracies_for_mlp_with_augmentations, test_loader, CFG.device, type='MLP', aug=True))
final_results.extend(print_ensemble_results(trained_models_for_cnn_with_augmentations, accuracies_for_cnn_with_augmentations, test_loader, CFG.device, type='CNN', aug=True))
final_results.extend(print_ensemble_results(trained_models_for_mlp_without_augmentations, accuracies_for_mlp_without_augmentations, test_loader, CFG.device, type='MLP', aug=False))
final_results.extend(print_ensemble_results(trained_models_for_cnn_without_augmentations, accuracies_for_cnn_without_augmentations, test_loader, CFG.device, type='CNN', aug=False))

results_df = pd.DataFrame(final_results)
cols = ['Model Type', 'Augmentation', 'Ensemble Size', 'Method', 'Accuracy', 'F1_Score']
results_df = results_df[cols]

print("\nEnsemble Results Summary:")
print(results_df)

results_df.to_csv('land_patches_ensemble_results_summary.csv', index=False)
print("Saved ensemble results to land_patches_ensemble_results_summary.csv")